# Validation harness

The real task trains on 2025-01-01..2025-10-31 and predicts the following two
months. Any local estimate of model quality has to have that same shape, or it
measures the wrong thing entirely — a random shuffle-split would let the model
see September while being tested on July, which is not the situation it will
actually face.

This notebook exercises `src/evaluation.py` on its own, before any real model
exists, to answer three questions:

1. Do the rolling-origin folds actually respect time order?
2. Does the metrics split (all rows vs. corrupted-excluded) behave the way the
   design intends?
3. What score does the simplest possible baseline get, so later models have
   something concrete to beat?

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import clean, load_raw
from src.evaluation import cross_validate, describe_folds, rolling_origin_folds

pd.set_option("display.width", 160)

train, _ = clean(load_raw("train_test.csv"))
folds = rolling_origin_folds(train.date)
describe_folds(folds, train.date)

,fold,train,train_rows,test,test_rows,gap_days
0,1,2025-01-01 -> 2025-07-01,28941,2025-07-02 -> 2025-08-31,9536,1
1,2,2025-01-01 -> 2025-07-31,33718,2025-08-01 -> 2025-09-30,9429,1
2,3,2025-01-01 -> 2025-08-31,38477,2025-09-01 -> 2025-10-31,9523,1


Three folds, each with an expanding training window and a fixed 61-day test
horizon — the same length as the real November-December forecast target. Only
the cutoff date moves between folds; training always starts on day one, mirroring
how the real model will be trained on all available history.

## 2. Leakage guard

In [2]:
for fold in folds:
    train_mask, test_mask = fold.masks(train.date)
    last_train = train.date[train_mask].max()
    first_test = train.date[test_mask].min()
    overlap = int((train_mask & test_mask).sum())
    print(f"fold {fold.index}: last train {last_train.date()} < first test {first_test.date()} "
          f"-> {last_train < first_test}   overlapping rows={overlap}")

fold 1: last train 2025-07-01 < first test 2025-07-02 -> True   overlapping rows=0
fold 2: last train 2025-07-31 < first test 2025-08-01 -> True   overlapping rows=0
fold 3: last train 2025-08-31 < first test 2025-09-01 -> True   overlapping rows=0


Every fold passes: the last training date is strictly before the first test
date, and zero rows belong to both sets. This is checked directly rather than
assumed from the date-range arithmetic, because a silent off-by-one here would
invalidate every score produced afterward.

## 3. Baseline: a single global $/mile

The simplest model that is not obviously broken: ignore every feature except
distance, and price every load at the training set's median $/mile. Any real
model needs to beat this by a meaningful margin to justify its complexity.

In [3]:
def flat_rate_per_mile(train_frame, test_frame):
    rate = (train_frame.posted_rate / train_frame.distance).median()
    return test_frame.distance * rate

baseline = cross_validate(train, flat_rate_per_mile)
baseline[["fold", "test_rows", "all_rmse", "all_mae", "all_median_ape",
         "clean_rmse", "clean_mae", "clean_median_ape", "corrupted_in_test"]].round(4)

,fold,test_rows,all_rmse,all_mae,all_median_ape,clean_rmse,clean_mae,clean_median_ape,corrupted_in_test
0,1,9536,665.395992,248.499486,0.078012,291.100739,199.834071,0.076839,0.013737
1,2,9429,674.134337,261.286226,0.082125,315.07903,213.510813,0.080791,0.013999
2,3,9523,684.247911,256.952939,0.080319,295.927042,203.68657,0.079051,0.015121
3,mean,9496.0,674.592747,255.57955,0.080152,300.70227,205.677152,0.078894,0.014286


**Two findings, not just plumbing.**

`all_rmse` is more than double `clean_rmse` (roughly 675 vs 300) while `all_mae`
and `clean_mae` are much closer (roughly 256 vs 206). RMSE squares each error
before averaging, so the ~677 corrupted loads dominate it disproportionately —
about half inflated to roughly 3x their peer-group price, half deflated to
roughly a third of it — and both directions produce a large squared error.
`all_rmse` is mostly reporting how much corruption happened to land in that
fold's test window, not how good the baseline is. **MAE and median APE are the
metrics worth trusting when comparing models.**

`corrupted_in_test` ranges from 1.37% to 1.51% across the three folds,
consistent with the 1.41% found across the whole training set — the corruption
is spread evenly over time, not concentrated in any one period.

## 4. Does excluding corrupted rows from training change anything here?

In [4]:
def flat_rate_clean(train_frame, test_frame):
    kept = train_frame[~train_frame.is_corrupted]
    rate = (kept.posted_rate / kept.distance).median()
    return test_frame.distance * rate

excluded = cross_validate(train, flat_rate_clean)
comparison = pd.concat(
    [baseline.set_index("fold")[["clean_mae", "clean_median_ape"]].add_prefix("all_rows_"),
     excluded.set_index("fold")[["clean_mae", "clean_median_ape"]].add_prefix("excl_corrupted_")],
    axis=1,
)
comparison.round(4)

,all_rows_clean_mae,all_rows_clean_median_ape,excl_corrupted_clean_mae,excl_corrupted_clean_median_ape
fold,,,,
1,199.834071,0.076839,199.844078,0.076837
2,213.510813,0.080791,213.486883,0.080813
3,203.68657,0.079051,203.685481,0.07905
mean,205.677152,0.078894,205.672148,0.0789


**No meaningful difference.** Both versions fit a *median*, and a median is
already insensitive to a small fraction of extreme outliers by construction —
whether or not those 677 corrupted rows are in the training set barely moves
where the median lands.

This is a useful negative result to have on record before building the real
model: **excluding corrupted rows only pays off once the model is sensitive to
outliers** (e.g. anything trained with squared error). It also confirms the
harness itself is not silently favoring one training set over the other for
some unrelated reason — the near-identical scores are exactly what should
happen when the only difference is a handful of already-outlier-robust rows.

## Summary

* Folds are leak-free by direct test, not by assumption.
* RMSE is unreliable here because of the injected label corruption; MAE and
  median APE are the metrics to report and compare going forward.
* Baseline to beat: **~$206 MAE, ~7.9% median APE** on clean rows.
* Excluding corrupted rows from training matters for outlier-sensitive models,
  not for the median baseline — worth re-checking once a real model exists.

